In [ ]:
#FILTRO 5-CORE
import pandas as pd

df = pd.read_csv(r"C:\TESI\Books.csv.gz", compression="gzip")
user_counts = df["user_id"].value_counts()
item_counts = df["parent_asin"].value_counts()

valid_users = set(user_counts[user_counts >= 5].index)
valid_items = set(item_counts[item_counts >= 5].index)

print(len(valid_users), "valid users")
print(len(valid_items), "valid items")

776370 valid users
495063 valid items


In [4]:
import json

input_file = r"C:\TESI\Books_test.jsonl\Books.jsonl"
output_file = r"C:\TESI\Books_5core\Books_5core.jsonl"

with open(input_file, "r", encoding="utf-8") as f, \
     open(output_file, "w", encoding="utf-8") as out:

    for line in f:
        row = json.loads(line)

        if row["user_id"] in valid_users and row["parent_asin"] in valid_items:
            out.write(json.dumps(row) + "\n")

In [ ]:
file_path = r"C:\TESI\Books_5core\Books_5core.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(json.loads(f.readline()))

{'rating': 5.0, 'title': 'Updated: after 1st arrived damaged this one is perfect', 'text': 'Updated: after first book arrived very damaged the replacement arrived in perfect condition.', 'images': [], 'asin': '0593235657', 'parent_asin': '0593235657', 'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'timestamp': 1640629604904, 'helpful_vote': 1, 'verified_purchase': True}
{'rating': 5.0, 'title': 'Excellent! I love it!', 'text': 'I bought it for the bag on the front so it paid for itself with that imo.  I haven’t started anything yet from it bc I’m still busy with other projects, but I’m really looking forward to starting some.', 'images': [], 'asin': '1782490671', 'parent_asin': '1782490671', 'user_id': 'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ', 'timestamp': 1640383495102, 'helpful_vote': 0, 'verified_purchase': True}
{'rating': 5.0, 'title': 'Beautiful patterns!', 'text': 'I love this book!  The patterns are lovely. I admit I’m not even following all of them to the letter. I am more of a free spirit.

In [6]:
from collections import Counter

user_counter = Counter()
item_counter = Counter()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        user_counter[row["user_id"]] += 1
        item_counter[row["parent_asin"]] += 1

print("min review per user:", min(user_counter.values()))
print("min review per item:", min(item_counter.values()))

min review per user: 5
min review per item: 5


In [7]:
print("numero totale review:", sum(user_counter.values()))
print("numero utenti:", len(user_counter))
print("numero prodotti:", len(item_counter))

numero totale review: 9564058
numero utenti: 776370
numero prodotti: 495063


In [ ]:
#CREAZIONE SUBSET DI 10K REVIEWS
input_file = r"C:\TESI\Books_5core\Books_5core.jsonl"
output_file = r"C:\TESI\Books_5core\Books_5core_subset_10k.jsonl"

n = 10000
count = 0

with open(input_file, "r", encoding="utf-8") as f, \
     open(output_file, "w", encoding="utf-8") as out:

    for line in f:
        out.write(line)
        count += 1
        
        if count >= n:
            break

print("subset creato con", count, "review")

subset creato con 10000 review


In [9]:
file_path = r"C:\TESI\Books_5core\Books_5core_subset_10k.jsonl"

count = 0

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        count += 1

print("numero review:", count)

numero review: 10000


In [ ]:
df = pd.read_json(
    r"C:\TESI\Books_5core\Books_5core_subset_10k.jsonl",
    lines=True
)

df_model = df[["text","parent_asin","rating"]].copy()

# rinomina colonne 
df_model = df_model.rename(columns={
    "text":"sentence",
    "parent_asin":"participant",
    "rating":"target"
})

df_model["item"] = df_model.index

# convertiamo categorie in indici interi
df_model["participant"] = df_model["participant"].astype("category").cat.codes
df_model["item"] = df_model["item"].astype("category").cat.codes

df_model["target"] = df_model["target"].map({
    1: 0,
    2: 0,
    3: 1,
    4: 2,
    5: 2
})

# salva dataset
df_model.to_csv(
    r"C:\TESI\amazon_model_dataset_10k.csv",
    index=False
)

print("dataset pronto:", df_model.shape)

dataset pronto: (10000, 4)


In [ ]:
df = pd.read_csv(r"C:\TESI\amazon_model_dataset_10k.csv")
print(df.columns)
print(df_model["target"].value_counts())
print(df.head())

Index(['sentence', 'participant', 'target', 'item'], dtype='object')
target
2    8403
1    1095
0     502
Name: count, dtype: int64
                                            sentence  participant  target  \
0  Updated: after first book arrived very damaged...         2661       2   
1  I bought it for the bag on the front so it pai...         7739       2   
2  I love this book!  The patterns are lovely. I ...         3800       2   
3  Seriously one of only a few books they I have ...         7297       2   
4  I love this book.  I was not blessed with arti...         7747       2   

   item  
0     0  
1     1  
2     2  
3     3  
4     4  


In [ ]:
import torch
emb = torch.load(r"C:\TESI\embeddings.pt")
print(emb.shape)

torch.Size([10000, 768])


In [ ]:
file_path = r"C:\TESI\Books_5core\Books_5core.jsonl"

item_counts = {}

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        item = row["parent_asin"]
        item_counts[item] = item_counts.get(item, 0) + 1

item_counts = pd.Series(item_counts)

In [2]:
threshold = item_counts.quantile(0.5)

top_items = set(item_counts[item_counts >= threshold].index)

print("Numero prodotti totali:", len(item_counts))
print("Numero prodotti selezionati:", len(top_items))

Numero prodotti totali: 495063
Numero prodotti selezionati: 269418


In [3]:
input_file = r"C:\TESI\Books_5core\Books_5core.jsonl"
output_file = r"C:\TESI\Books_5core\Books_top50.jsonl"

count_total = 0
count_kept = 0

with open(input_file, "r", encoding="utf-8") as f, \
     open(output_file, "w", encoding="utf-8") as out:

    for line in f:
        count_total += 1
        row = json.loads(line)

        if row["parent_asin"] in top_items:
            out.write(json.dumps(row) + "\n")
            count_kept += 1

print("Review totali:", count_total)
print("Review dopo filtro:", count_kept)

Review totali: 9564058
Review dopo filtro: 8172610


In [3]:
#ordina prodotti per numero di review (decrescente)
item_counts_sorted = item_counts.sort_values(ascending=False)

#cumulata delle review
cumulative_reviews = item_counts_sorted.cumsum()

#trova indice dove superi 400k
target_reviews = 400_000

cutoff_idx = cumulative_reviews.searchsorted(target_reviews)

#seleziona prodotti necessari
selected_items = item_counts_sorted.iloc[:cutoff_idx + 1]

#risultati
num_products = len(selected_items)
total_products = len(item_counts_sorted)

perc_products = num_products / total_products * 100
total_reviews = selected_items.sum()

print(f"Numero prodotti selezionati: {num_products}")
print(f"Percentuale prodotti: {perc_products:.2f}%")
print(f"Numero review totali: {total_reviews}")

Numero prodotti selezionati: 234
Percentuale prodotti: 0.05%
Numero review totali: 400169
